In [2]:
reviews = [
    "I absolutely love this product! It works like a charm.",
    "This is the worst purchase I have ever made.",
    "It's okay, not great but not terrible either."
]


text = ["Oh wow, CircaSum’s ‘lightning-fast’ support took only 72 hours to reply with a copy-pasted FAQ link! Truly groundbreaking service. 10/10 would recommend... if you love frustration."]
 

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline #pytorch

from transformers import TFAutoModelForCausalLM   #tensorflow

import torch
import torch.nn.functional as F
from torch.nn.functional import softmax

c:\Users\nikky\anaconda3\envs\rakaenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\nikky\anaconda3\envs\rakaenv\lib\site-packages\torch\nn\modules\transformer.py:20: UserWarning: Failed to initialize NumPy: DLL load failed while importing _multiarray_umath: The specified module could not be found. (Triggered internally at ..\torch\csrc\utils\tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


In [4]:

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
sentiment_analysis_model = "sentiment-analysis"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


In [5]:
encoder = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
print(encoder)

{'input_ids': tensor([[  101, 17003, 94608,   117, 13534, 24667,   100,   161,   100, 38241,
           118, 14024,   100, 13016, 12384, 10902, 12294, 18030, 10114, 37090,
         10563, 10171,   143, 35394,   118, 15904, 10390, 18044, 17266,   106,
         69434, 15316, 83330, 10285, 11416,   119, 10148,   120, 10148, 11008,
         44909, 55667, 10163,   119,   119,   119, 11526, 10855, 11157, 72642,
         53984, 11210,   119,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])}


In [6]:
print(encoder["input_ids"][0])
print(encoder["attention_mask"][0])
print(encoder["input_ids"].shape)
print(encoder["attention_mask"][0])

tensor([  101, 17003, 94608,   117, 13534, 24667,   100,   161,   100, 38241,
          118, 14024,   100, 13016, 12384, 10902, 12294, 18030, 10114, 37090,
        10563, 10171,   143, 35394,   118, 15904, 10390, 18044, 17266,   106,
        69434, 15316, 83330, 10285, 11416,   119, 10148,   120, 10148, 11008,
        44909, 55667, 10163,   119,   119,   119, 11526, 10855, 11157, 72642,
        53984, 11210,   119,   102])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])
torch.Size([1, 54])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])


In [7]:
oupputs = model(**encoder)
print(oupputs)
print(oupputs.logits)
print(oupputs.logits.shape)

SequenceClassifierOutput(loss=None, logits=tensor([[-0.2824, -0.6976, -0.8104,  0.1264,  1.2943]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[-0.2824, -0.6976, -0.8104,  0.1264,  1.2943]],
       grad_fn=<AddmmBackward0>)
torch.Size([1, 5])


In [8]:
#convert logits to probabilities
prob = F.softmax(oupputs.logits, dim=1) # prob = F.softmax(oupputs.logits, dim=-1) #-1: automatically adjust last dimension
print(prob)

tensor([[0.1164, 0.0768, 0.0686, 0.1751, 0.5631]], grad_fn=<SoftmaxBackward0>)


In [9]:
for i, p in enumerate(prob[0]):
    print(f"Class {i}: {p.item():.4f}")

Class 0: 0.1164
Class 1: 0.0768
Class 2: 0.0686
Class 3: 0.1751
Class 4: 0.5631


In [10]:
#get model predictions
with torch.no_grad():
    predtictoons = model(**encoder)
    probabilities = F.softmax(predtictoons.logits, dim=1)
    pred_class = torch.argmax(probabilities, dim=1)

#map lable
map_label = {0: "Not Sarcastic", 1: "Sarcastic"}

print("Predicted class:", pred_class.item())
print("Probabilities:", probabilities)
for i, p in enumerate(pred_class):
    print(f"Class {i}: {map_label[i]} with probability {probabilities[0][i].item():.4f}")

Predicted class: 4
Probabilities: tensor([[0.1164, 0.0768, 0.0686, 0.1751, 0.5631]])
Class 0: Not Sarcastic with probability 0.1164


In [11]:

#using small model of t5
model_name = "t5-small"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Input text
text = "The service was fantastic and the food was great."

# Tokenize input
inputs = tokenizer(text, return_tensors="pt")

# Get model output
with torch.no_grad():
    outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=1)

print("probs : ", probs)

# Decode: get predicted class (highest probability)
predicted_class = torch.argmax(probs, dim=1).item()  # 0 to 4

print("Predicted class (0-4):", predicted_class)

# Convert to human-readable label
star_rating = predicted_class + 1  # Because classes are 0-indexed

# Optional: add sentiment label
sentiment_labels = {
    1: "Very Negative",
    2: "Negative",
    3: "Neutral",
    4: "Positive",
    5: "Very Positive"
}

print(f"Predicted Rating: {star_rating} star ({sentiment_labels[star_rating]})")


Some weights of T5ForSequenceClassification were not initialized from the model checkpoint at t5-small and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


probs :  tensor([[0.6780, 0.3220]])
Predicted class (0-4): 0
Predicted Rating: 1 star (Very Negative)


In [12]:
texts = [
    "This was the worst experience I've ever had.",             # 1 star - Very Negative
    "I didn't like the food at all.",                           # 2 stars - Negative
    "It was okay, nothing special.",                            # 3 stars - Neutral
    "Pretty good overall, I would recommend it.",               # 4 stars - Positive
    "Absolutely amazing! I loved everything about it.",         # 5 stars - Very Positive
    "The product broke after one use. Terrible quality.",       # 1 star - Very Negative
    "Service was slow and the staff was rude.",                 # 2 stars - Negative
    "Mediocre taste but decent portion size.",                  # 3 stars - Neutral
    "Tasty, affordable, and quick. I'd come back again.",       # 4 stars - Positive
    "Best restaurant I've visited this year!",                  # 5 stars - Very Positive
]


In [13]:
c1 = 0
c2 = 0
for text in texts:
    c1 = c1 + 1
    inputs = tokenizer(text, return_tensors="pt")
    print("counter 1 : ", c1)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item() + 1  # convert 0-indexed to 1-5

        print("probs : ", probs)
        print("Predicted class (0-4):", predicted_class)

        label = sentiment_labels[predicted_class]
        c2 = c2 + 1
        print("counter 2 : ", c2)
        print(f"Text: {text}\n→ Predicted: {predicted_class} star ({label})\n")
        print("#"*50)


counter 1 :  1
probs :  tensor([[0.5782, 0.4218]])
Predicted class (0-4): 1
counter 2 :  1
Text: This was the worst experience I've ever had.
→ Predicted: 1 star (Very Negative)

##################################################
counter 1 :  2
probs :  tensor([[0.5761, 0.4239]])
Predicted class (0-4): 1
counter 2 :  2
Text: I didn't like the food at all.
→ Predicted: 1 star (Very Negative)

##################################################
counter 1 :  3
probs :  tensor([[0.5902, 0.4098]])
Predicted class (0-4): 1
counter 2 :  3
Text: It was okay, nothing special.
→ Predicted: 1 star (Very Negative)

##################################################
counter 1 :  4
probs :  tensor([[0.5750, 0.4250]])
Predicted class (0-4): 1
counter 2 :  4
Text: Pretty good overall, I would recommend it.
→ Predicted: 1 star (Very Negative)

##################################################
counter 1 :  5
probs :  tensor([[0.6165, 0.3835]])
Predicted class (0-4): 1
counter 2 :  5
Text: Absolutely ama

In [15]:
from transformers import pipeline

# Load the sentiment analysis pipeline with the custom model
sentiment_pipeline = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

# Example usage
text = "I love this product, it's absolutely fantastic!"
result = sentiment_pipeline(text)

print(result)


Device set to use cpu


RuntimeError: Numpy is not available